In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import csv




I0 = 9.047e-01  
out_csv = "prepared_data.csv"
fig1_path = "ib_curve.png"
fig2_path = "linearized_view.png"
dpi = 300




points = np.arange(1, 17, dtype=int)
G = np.array([9.630e-001,4.045e+000,7.126e+000,1.021e+001,1.329e+001,1.637e+001,
              1.945e+001,2.253e+001,2.562e+001,2.870e+001,3.178e+001,3.486e+001,
              3.794e+001,4.102e+001,4.410e+001,4.719e+001], dtype=float)  
Expt = np.array([1.000e+000,9.509e-001,8.792e-001,7.926e-001,7.150e-001,6.369e-001,
                 5.676e-001,4.994e-001,4.386e-001,3.851e-001,3.375e-001,2.988e-001,
                 2.600e-001,2.286e-001,2.031e-001,1.794e-001], dtype=float)





b = 3.825e8 * G**2


ln_Expt_over_I0 = np.log(Expt / I0)





def bi_exp_fixed_I0(b, f, D_fast, D_slow):
    
    f = np.clip(f, 1e-6, 1-1e-6)
    D_fast = max(D_fast, 1e-14)
    D_slow = max(D_slow, 1e-14)
    if D_fast < D_slow:
        D_fast, D_slow = D_slow, D_fast
        f = 1 - f
    return I0 * ( f*np.exp(-D_fast*b) + (1-f)*np.exp(-D_slow*b) )


p0 = [0.4, 1.0e-11, 1.0e-12]
bounds = ([1e-6, 1e-14, 1e-14], [1-1e-6, 1e-9, 1e-9])


popt, pcov = curve_fit(bi_exp_fixed_I0, b, Expt, p0=p0, bounds=bounds, maxfev=300000)
f_fit, D_fast_fit, D_slow_fit = popt
perr = np.sqrt(np.diag(pcov))
f_err, D_fast_err, D_slow_err = perr


fit = bi_exp_fixed_I0(b, f_fit, D_fast_fit, D_slow_fit)
resid = Expt - fit
rss = float(np.sum(resid**2))
tss = float(np.sum((Expt - np.mean(Expt))**2))
r2 = 1 - rss/tss


print("Two-component diffusion fit (I0 fixed at 0.9047)")
print(f"I0           = {I0:.6f} (fixed)")
print(f"f_fast       = {f_fit:.4f} ± {f_err:.4f}")
print(f"D_fast       = {D_fast_fit:.3e} ± {D_fast_err:.3e} m^2/s")
print(f"D_slow       = {D_slow_fit:.3e} ± {D_slow_err:.3e} m^2/s")
print(f"RSS          = {rss:.6e}")
print(f"R^2          = {r2:.6f}")




with open(out_csv, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Point","G(G/cm)","b(s·m^-2)","Expt","ln(Expt/I0)"])
    for i in range(len(points)):
        writer.writerow([int(points[i]), f"{G[i]:.6e}", f"{b[i]:.6e}", f"{Expt[i]:.6e}", f"{ln_Expt_over_I0[i]:.6e}"])
print(f"Saved CSV -> {out_csv}")





b_dense = np.linspace(0, b.max()*1.05, 800)
fit_dense = bi_exp_fixed_I0(b_dense, f_fit, D_fast_fit, D_slow_fit)


plt.figure(figsize=(6.2, 4.6))
plt.scatter(b, Expt, color="#333333", s=28, label="Expt")
plt.plot(b_dense, fit_dense, color="#D81B60", lw=2.2, label="Bi-exp fit")
plt.xlabel("b (s·m$^{-2}$)")
plt.ylabel("I")
plt.title("I vs b (Two-component fit, $I_0$ fixed)")
plt.legend(frameon=False)
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.savefig(fig1_path, dpi=dpi)
plt.close()
print(f"Saved figure -> {fig1_path}")


plt.figure(figsize=(6.2, 4.6))
plt.scatter(b, ln_Expt_over_I0, color="#333333", s=28, label="ln(Expt/I0)")
plt.plot(b_dense, np.log(bi_exp_fixed_I0(b_dense, f_fit, D_fast_fit, D_slow_fit)/I0),
         color="#1E88E5", lw=2.2, label="Model (bi-exp)")
plt.xlabel("b (s·m$^{-2}$)")
plt.ylabel("ln(I/I0)")
plt.title("Linearized view (Double-exponential model)")
plt.legend(frameon=False)
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.savefig(fig2_path, dpi=dpi)
plt.close()
print(f"Saved figure -> {fig2_path}")